# 02 - Create Grid and Process NDVI

This notebook creates a regular spatial grid over Morocco and Western Sahara, then extracts mean MODIS NDVI for each grid cell and each MODIS 16-day raster. The input files are `.tif` GeoTIFFs in `data/raw/ndvi_2023/`; this notebook creates the output CSV `data/processed/ndvi_grid_2023.csv`. NDVI extraction can take time in Colab because every raster is summarized over every grid cell.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Edit this path to the location where you upload the project folder in Google Drive.
PROJECT_DIR = "/content/drive/MyDrive/fire-risk-project"
PROJECT_DIR = Path(PROJECT_DIR)

NDVI_DIR = PROJECT_DIR / "data" / "raw" / "ndvi_2023"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GRID_OUTPUT_PATH = PROCESSED_DIR / "morocco_ws_grid.geojson"
NDVI_OUTPUT_PATH = PROCESSED_DIR / "ndvi_grid_2023.csv"

print("NDVI_DIR:", NDVI_DIR)
print("GRID_OUTPUT_PATH:", GRID_OUTPUT_PATH)
print("NDVI_OUTPUT_PATH:", NDVI_OUTPUT_PATH)


Mounted at /content/drive
NDVI_DIR: /content/drive/MyDrive/fire-risk-project/data/raw/ndvi_2023
GRID_OUTPUT_PATH: /content/drive/MyDrive/fire-risk-project/data/processed/morocco_ws_grid.geojson
NDVI_OUTPUT_PATH: /content/drive/MyDrive/fire-risk-project/data/processed/ndvi_grid_2023.csv


In [2]:
import re

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterstats import zonal_stats
from shapely.geometry import box
from tqdm.auto import tqdm


In [3]:
WEST, SOUTH, EAST, NORTH = -17.2, 20.5, -1.0, 36.2
GRID_SIZE = 0.25

def create_bbox_grid(west, south, east, north, grid_size):
    records = []
    grid_id = 0
    y = south
    while y < north:
        x = west
        y2 = min(y + grid_size, north)
        while x < east:
            x2 = min(x + grid_size, east)
            records.append(
                {
                    "grid_id": grid_id,
                    "west": x,
                    "south": y,
                    "east": x2,
                    "north": y2,
                    "geometry": box(x, y, x2, y2),
                }
            )
            grid_id += 1
            x = round(x + grid_size, 10)
        y = round(y + grid_size, 10)
    return gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")


grid = create_bbox_grid(WEST, SOUTH, EAST, NORTH, GRID_SIZE)
grid.to_file(GRID_OUTPUT_PATH, driver="GeoJSON")
print(f"Created {len(grid):,} grid cells")
display(grid.head())


Created 4,095 grid cells


,grid_id,west,south,east,north,geometry
0,0,-17.20,20.5,-16.95,20.75,"POLYGON ((-16.95 20.5, -16.95 20.75, -17.2 20...."
1,1,-16.95,20.5,-16.70,20.75,"POLYGON ((-16.7 20.5, -16.7 20.75, -16.95 20.7..."
2,2,-16.70,20.5,-16.45,20.75,"POLYGON ((-16.45 20.5, -16.45 20.75, -16.7 20...."
3,3,-16.45,20.5,-16.20,20.75,"POLYGON ((-16.2 20.5, -16.2 20.75, -16.45 20.7..."
4,4,-16.20,20.5,-15.95,20.75,"POLYGON ((-15.95 20.5, -15.95 20.75, -16.2 20...."


In [4]:
all_tif_files = sorted(NDVI_DIR.glob("*.tif"))
ndvi_files = sorted(
    p for p in NDVI_DIR.glob("*.tif")
    if re.search(r"_NDVI_\d{8}T\d{6}", p.name)
)
quality_files = sorted(NDVI_DIR.glob("*VI_Quality*.tif"))

print(f"Found {len(all_tif_files)} total GeoTIFF files")
print(f"Found {len(ndvi_files)} NDVI GeoTIFF files")
print(f"Ignoring {len(quality_files)} VI quality GeoTIFF files for now")

if not ndvi_files:
    raise FileNotFoundError(f"No NDVI GeoTIFF files matching '*NDVI*.tif' were found in {NDVI_DIR}")

bad_files = [p.name for p in ndvi_files if "VI_Quality" in p.name]
assert not bad_files, f"Quality rasters were incorrectly included: {bad_files[:5]}"

for path in ndvi_files[:10]:
    print(path.name)


Found 48 total GeoTIFF files
Found 24 NDVI GeoTIFF files
Ignoring 24 VI quality GeoTIFF files for now
MOD13Q1.061__250m_16_days_NDVI_20221219T000000_aid0001.tif
MOD13Q1.061__250m_16_days_NDVI_20230101T000000_aid0001.tif
MOD13Q1.061__250m_16_days_NDVI_20230117T000000_aid0001.tif
MOD13Q1.061__250m_16_days_NDVI_20230202T000000_aid0001.tif
MOD13Q1.061__250m_16_days_NDVI_20230218T000000_aid0001.tif
MOD13Q1.061__250m_16_days_NDVI_20230306T000000_aid0001.tif
MOD13Q1.061__250m_16_days_NDVI_20230322T000000_aid0001.tif
MOD13Q1.061__250m_16_days_NDVI_20230407T000000_aid0001.tif
MOD13Q1.061__250m_16_days_NDVI_20230423T000000_aid0001.tif
MOD13Q1.061__250m_16_days_NDVI_20230509T000000_aid0001.tif


In [5]:
def extract_date_from_ndvi_filename(path):
    match = re.search(r"(\d{8})T\d{6}", Path(path).name)
    if not match:
        raise ValueError(f"Could not parse NDVI date from {path}")
    return pd.to_datetime(match.group(1), format="%Y%m%d")


if ndvi_files:
    print(ndvi_files[0].name, "->", extract_date_from_ndvi_filename(ndvi_files[0]))


MOD13Q1.061__250m_16_days_NDVI_20221219T000000_aid0001.tif -> 2022-12-19 00:00:00


In [6]:
# This cell may take time in Colab. It does not download data; it summarizes local GeoTIFFs already in NDVI_DIR.
records = []

for tif_path in tqdm(ndvi_files, desc="Extracting NDVI by grid"):
    ndvi_date = extract_date_from_ndvi_filename(tif_path)

    with rasterio.open(tif_path) as src:
        raster_crs = src.crs
        nodata = src.nodata

    # MOD13Q1 NDVI valid raw range is approximately -2000 to 10000.
    # AppEEARS files usually expose nodata, but if not, use the MODIS NDVI fill value.
    if nodata is None:
        nodata = -3000

    zones = grid.to_crs(raster_crs) if raster_crs and grid.crs != raster_crs else grid
    stats = zonal_stats(
        zones,
        str(tif_path),
        stats=["mean"],
        nodata=nodata,
        all_touched=True,
    )

    for grid_id, item in zip(grid["grid_id"], stats):
        records.append(
            {
                "grid_id": grid_id,
                "date": ndvi_date,
                "ndvi_raw_mean": item.get("mean"),
            }
        )

ndvi_grid = pd.DataFrame(records)
expected_rows = len(grid) * len(ndvi_files)
duplicate_rows = ndvi_grid.duplicated(["grid_id", "date"]).sum()
missing_raw = ndvi_grid["ndvi_raw_mean"].isna().sum()

print(ndvi_grid.shape)
print(f"Expected rows: {expected_rows:,}")
print(f"Duplicate grid/date rows: {duplicate_rows:,}")
print(f"Rows with no raster data for the grid cell: {missing_raw:,}")

if len(ndvi_grid) != expected_rows:
    print("Warning: row count differs from grid cells x NDVI files.")
if duplicate_rows:
    print("Warning: duplicate grid/date rows found. Check that only NDVI rasters were processed.")

display(ndvi_grid.head())


Extracting NDVI by grid:   0%|          | 0/24 [00:00<?, ?it/s]

(98280, 3)
Expected rows: 98,280
Duplicate grid/date rows: 0
Rows with no raster data for the grid cell: 31,292


,grid_id,date,ndvi_raw_mean
0,0,2022-12-19,NaN
1,1,2022-12-19,NaN
2,2,2022-12-19,882.984015
3,3,2022-12-19,948.193471
4,4,2022-12-19,971.227153


In [7]:
raw_invalid = (ndvi_grid["ndvi_raw_mean"] < -2000) | (ndvi_grid["ndvi_raw_mean"] > 10000)
print(f"Raw NDVI means outside MODIS valid range masked before scaling: {raw_invalid.sum():,}")
ndvi_grid.loc[raw_invalid, "ndvi_raw_mean"] = np.nan

ndvi_grid["ndvi"] = ndvi_grid["ndvi_raw_mean"].astype(float) * 0.0001
display(ndvi_grid.head())


Raw NDVI means outside MODIS valid range masked before scaling: 0


,grid_id,date,ndvi_raw_mean,ndvi
0,0,2022-12-19,NaN,NaN
1,1,2022-12-19,NaN,NaN
2,2,2022-12-19,882.984015,0.088298
3,3,2022-12-19,948.193471,0.094819
4,4,2022-12-19,971.227153,0.097123


In [8]:
invalid_mask = (ndvi_grid["ndvi"] < -0.2) | (ndvi_grid["ndvi"] > 1.0)
print(f"Invalid NDVI values masked: {invalid_mask.sum():,}")
ndvi_grid.loc[invalid_mask, "ndvi"] = np.nan


Invalid NDVI values masked: 0


In [9]:
ndvi_grid = ndvi_grid[["grid_id", "date", "ndvi_raw_mean", "ndvi"]].sort_values(["grid_id", "date"])
ndvi_grid.to_csv(NDVI_OUTPUT_PATH, index=False)
print(f"Saved NDVI grid table to {NDVI_OUTPUT_PATH}")
display(ndvi_grid.head())


Saved NDVI grid table to /content/drive/MyDrive/fire-risk-project/data/processed/ndvi_grid_2023.csv


,grid_id,date,ndvi_raw_mean,ndvi
0,0,2022-12-19,NaN,NaN
4095,0,2023-01-01,NaN,NaN
8190,0,2023-01-17,NaN,NaN
12285,0,2023-02-02,NaN,NaN
16380,0,2023-02-18,NaN,NaN
